# 2DTFIM 1DNQS - (Nx,Ny)=(8,8): Inference (seed 111)

This is part of the work arxiv: 2606.25600 (Two-dimensional Hyperbolic RNN Neural Quantum State). For the purpose of reproducing the results, please check the link to the trained weight files. The saved weight links used in this notebook might not be the same as the ones in the Github repo. 

In [1]:
import sys
import os
sys.path.append('../../utility_tfim')
from lorentz_tfim2d_1drnn_train_loop import *
from poincare_tfim2d_1drnn_train_loop import *
import time

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.


In [2]:
def set_cpu_deterministic(seed=111):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

set_cpu_deterministic(111)

In [3]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped 

def define_load_test(wf, numsamples,path_to_weights, Ee, clipped_e = False):
    test_samples_before = wf.sample(numsamples)
    print(f'The number of samples is {len(test_samples_before)}')
    # --- PART A: Check performance BEFORE loading (Baseline) ---
    wf.model.eval() 
    with torch.no_grad():
        test_gs_before = Ising2D_local_energies(Jz, Bx, Nx, Ny, test_samples_before, wf)
        gs_mean_b = np.round(np.mean(test_gs_before),4)
        gs_var_b = np.round(np.var(test_gs_before),4)
    print(f'Before loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_b}, var E = {gs_var_b}')
    print('====================================================================')

     # --- PART B: Remap and Load the Weights ---
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    # This line loads the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        test_samples_after = wf.sample(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = Ising2D_local_energies(Jz, Bx, Nx, Ny, test_samples_after, wf)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after = Ising2D_local_energies(Jz, Bx, Nx, Ny, test_samples_after, wf)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy (not exact in 2D) is {np.round(Ee,4)}')

In [4]:
Nx=8
Ny=8
Bx=3.0
units =60
Jz=np.ones((Nx,Ny))
nsamples = 10000
E_dmrg =  -202.5077381261612
seed=111
fname = f'../2DTFIM_1DNQS_results_seed_{seed}/(8,8)'

## RNN variants

In [8]:
wf = RNNwavefunction(Nx, Ny, 'EuclRNN', units, seed=seed)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Euclidean/EuclRNN_60_8x8_ns=80_rmax=None_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 3,902
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -192.1374, var E = 111.7753
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -196.6122, var E = 43.9726
DMRG energy (not exact in 2D) is -202.5077
Time taken =0.026 hrs


In [11]:
r_max=1.0 # Best among the 3: r_max = 0.65, 0.8, 1.0
wf= RNNwavefunction_hyp(Nx, Ny, cell_type='HypRNN',r_max=r_max, bias_geom='hyp',
                           hyp_non_lin='id', units=units, seed=111)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/PoincareRNN/HypRNN_Nx=8_60_8x8_ns=80_rmax=1.0_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 3,902
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -192.0265, var E = 110.9221
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -201.7627, var E = 7.2617
DMRG energy (not exact in 2D) is -202.5077
Time taken =0.109 hrs


In [12]:
sc=2.0 # best among sc=2.0,4.0, 6.0
wf= Lorentzwavefunction(systemsize_x=Nx, systemsize_y=Ny, cell_type='LorentzRNN', 
                        units=units, spatial_clamp=sc, seed=111)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/LorentzRNN/LorentzRNN_60_8x8_ns=80_spatial_cl={sc}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 3,902
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -192.0714, var E = 110.4875
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -201.5256, var E = 9.9281
DMRG energy (not exact in 2D) is -202.5077
Time taken =0.292 hrs


In [9]:
sc=6.0 
wf= Lorentzwavefunction(systemsize_x=Nx, systemsize_y=Ny, cell_type='LorentzRNN', 
                        units=units, spatial_clamp=sc, non_lin='id', seed=111)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/LorentzRNN/LorentzRNN_60_8x8_ns=80_spatial_cl={sc}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 3,902
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -192.036, var E = 110.8143
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -200.208, var E = 25.5232
DMRG energy (not exact in 2D) is -202.5077
Time taken =0.315 hrs


## GRU variants

In [14]:
wf = RNNwavefunction(Nx, Ny, 'EuclGRU', units, seed=seed)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Euclidean/EuclGRU_{units}_8x8_ns=80_rmax=None_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 11,462
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -190.1644, var E = 135.3325
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -200.9295, var E = 19.4065
DMRG energy (not exact in 2D) is -202.5077
Time taken =0.067 hrs


In [15]:
r_max=1.0 # Best among 1.0, 0.8, 0.65
wf= RNNwavefunction_hyp(Nx, Ny, cell_type='HypGRU',r_max=r_max, bias_geom='hyp',
                           hyp_non_lin='id', units=units, seed=111)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/PoincareGRU/HypGRU_Nx=8_60_8x8_ns=80_rmax=1.0_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 11,462
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -190.9275, var E = 123.2812
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -201.5483, var E = 11.0951
DMRG energy (not exact in 2D) is -202.5077
Time taken =0.413 hrs


In [16]:
sc=2.0 #better than 4.0, comparable to 6.0?
wf= Lorentzwavefunction(systemsize_x=Nx, systemsize_y=Ny, cell_type='LorentzGRU', 
                        units=units, spatial_clamp=sc, seed=111)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/LorentzGRU/LorentzGRU_60_8x8_ns=80_spatial_cl={sc}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 11,462
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -193.5173, var E = 94.9539
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -200.8296, var E = 19.8163
DMRG energy (not exact in 2D) is -202.5077
Time taken =0.98 hrs


In [5]:
sc=4.0
wf= Lorentzwavefunction(systemsize_x=Nx, systemsize_y=Ny, cell_type='LorentzGRU', 
                        units=units, spatial_clamp=sc, non_lin='id', seed=111)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/LorentzGRU/LorentzGRU_60_8x8_ns=80_spatial_cl={sc}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 11,462
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -191.8802, var E = 112.7036
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -200.8135, var E = 19.8259
DMRG energy (not exact in 2D) is -202.5077
Time taken =0.697 hrs


In [17]:
sc=6.0 #better than 4.0, comparable to 2.0?
wf= Lorentzwavefunction(systemsize_x=Nx, systemsize_y=Ny, cell_type='LorentzGRU', 
                        units=units, spatial_clamp=sc, seed=111)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/LorentzGRU/LorentzGRU_60_8x8_ns=80_spatial_cl={sc}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 11,462
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -193.5659, var E = 94.6429
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -200.8363, var E = 19.9592
DMRG energy (not exact in 2D) is -202.5077
Time taken =0.952 hrs
